In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import string
import os
import re
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

BASE_URL = "https://www.mayoclinic.org"
SAVE_DIR = "/content/drive/MyDrive/capstone/data"
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

TARGET_SECTIONS = ["Symptoms", "Causes", "Risk factors"]
TARGET_SET = frozenset(TARGET_SECTIONS)

SKIP_CLASSES = [
    "eloqua", "subscription", "requestappt", "modal",
    "dialog", "ad-container", "acces-list",
    "contentbox", "rc-list", "related-content",
    "vertical-links", "dynamic-related-content",
]

STOP_H2_CLASSES = [
    "color-primary-inverse",
    "related-content",
    "vertical-links",
    "dynamic-related-content",
]

MAX_WORKERS = 8
REQUEST_DELAY = 0.15

_session = requests.Session()
_session.headers.update(HEADERS)
_adapter = requests.adapters.HTTPAdapter(
    pool_connections=MAX_WORKERS,
    pool_maxsize=MAX_WORKERS,
    max_retries=2,
)
_session.mount("https://", _adapter)
_session.mount("http://", _adapter)

_cache = {}
_cache_lock = threading.Lock()


def get_disease_links(letter):
    url = f"{BASE_URL}/diseases-conditions/index?letter={letter}"
    resp = _session.get(url, timeout=30)
    soup = BeautifulSoup(resp.text, "html.parser")

    links = []
    seen = set()

    def _add(name, href):
        if not name or not href:
            return
        full = href if href.startswith("http") else BASE_URL + href
        if "/diseases-conditions/" not in full or "/index" in full:
            return
        name = name.strip().rstrip("--- -").strip()
        if not name:
            return
        key = (name.lower(), full)
        if key not in seen:
            seen.add(key)
            links.append({"name": name, "url": full})

    for el in soup.find_all(class_=re.compile(r"primary-name")):
        name_text = None
        for span in el.find_all("span"):
            cls_s = " ".join(span.get("class", []))
            if "separator" in cls_s:
                continue
            if "__name" in cls_s:
                name_text = span.get_text(strip=True)
                break
        see_a = None
        for a in el.find_all("a", href=True):
            a_cls = " ".join(a.get("class", []))
            a_txt = a.get_text(strip=True)
            if "see" in a_cls.lower() or a_txt.startswith("See"):
                see_a = a
                break
        if name_text and see_a:
            _add(name_text, see_a["href"])

    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/diseases-conditions/" not in href or "/index" in href:
            continue
        text = a.get_text(strip=True)
        cls_s = " ".join(a.get("class", []))
        if "see-link" in cls_s or "see_link" in cls_s or text.startswith("See "):
            target = text[4:].strip() if text.startswith("See ") else text
            _add(target, href)
            continue
        _add(text, href)

    return links


def _has_skip_ancestor(elem):
    p = elem.parent
    while p and p.name:
        cls_lower = " ".join(p.get("class", [])).lower()
        if cls_lower and any(s in cls_lower for s in SKIP_CLASSES):
            return True
        p = p.parent
    return False


def extract_section_text(h2_tag):
    parts, seen = [], set()
    for elem in h2_tag.find_all_next():
        if elem.name == "h2":
            if elem.get_text(strip=True) in TARGET_SET:
                break
            cls = " ".join(elem.get("class", []))
            if any(f in cls for f in STOP_H2_CLASSES):
                break
            continue
        if elem.name in ("button", "script", "style", "nav", "footer",
                          "header", "input", "label", "form", "select",
                          "svg", "img", "iframe", "noscript"):
            continue
        if elem.name not in ("p", "h3", "h4", "ul", "ol"):
            continue
        if _has_skip_ancestor(elem):
            continue
        txt = elem.get_text(strip=True)
        if txt and txt not in seen:
            seen.add(txt)
            parts.append(txt)
    return " | ".join(parts)


def scrape_disease(url):
    with _cache_lock:
        if url in _cache:
            return _cache[url]
    try:
        resp = _session.get(url, timeout=30, allow_redirects=True)
        time.sleep(REQUEST_DELAY)
        if resp.status_code != 200:
            with _cache_lock:
                _cache[url] = None
            return None
    except requests.RequestException:
        with _cache_lock:
            _cache[url] = None
        return None

    soup = BeautifulSoup(resp.text, "html.parser")
    h1 = soup.find("h1")
    title = h1.get_text(strip=True) if h1 else ""
    if not title:
        with _cache_lock:
            _cache[url] = None
        return None

    data = {s: "" for s in TARGET_SECTIONS}
    for h2 in soup.find_all("h2"):
        heading = h2.get_text(strip=True)
        if heading not in TARGET_SET:
            continue
        cls = " ".join(h2.get("class", []))
        if any(f in cls for f in STOP_H2_CLASSES):
            continue
        text = extract_section_text(h2)
        if text and not data[heading]:
            data[heading] = text

    result = {
        "disease_name": title,
        "symptoms":     data["Symptoms"],
        "causes":       data["Causes"],
        "risk_factors": data["Risk factors"],
    }
    with _cache_lock:
        _cache[url] = result
    return result


def _scrape_one(link):
    result = scrape_disease(link["url"])
    return link, result


os.makedirs(SAVE_DIR, exist_ok=True)

print("Collecting disease links...")
all_links = []
for letter in tqdm(string.ascii_uppercase, desc="Indexing"):
    all_links.extend(
        [{"letter": letter, **d} for d in get_disease_links(letter)]
    )
    time.sleep(0.1)
print(f"Total entries found: {len(all_links)}")

unique_urls = set()
unique_links = []
for link in all_links:
    if link["url"] not in unique_urls:
        unique_urls.add(link["url"])
        unique_links.append(link)

print(f"Scraping {len(unique_links)} unique pages...")

results_map = {}
failed_urls = set()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(_scrape_one, lk): lk for lk in unique_links}
    with tqdm(total=len(futures), desc="Scraping") as pbar:
        for fut in as_completed(futures):
            link, result = fut.result()
            if result and result["disease_name"]:
                results_map[link["url"]] = result
            else:
                failed_urls.add(link["url"])
            pbar.update(1)

all_results = []
for link in all_links:
    result = results_map.get(link["url"])
    if result:
        entry = result.copy()
        all_results.append(entry)

df = pd.DataFrame(all_results) if all_results else pd.DataFrame()
df = df.drop_duplicates(subset="disease_name")
output_path = os.path.join(SAVE_DIR, "mayo_clinic_diseases_raw.csv")
df.to_csv(output_path, index=False)

print(f"Done. Saved {len(df)} unique diseases to {output_path}")
print(f"Columns: {df.columns.tolist()}")
print(f"Failed pages: {len(failed_urls)}")
for col in ["symptoms", "causes", "risk_factors"]:
    filled = (df[col] != "").sum()
    print(f"  {col}: {filled}/{len(df)} filled")

Indexing: 100%|██████████| 26/26 [00:11<00:00,  2.23it/s]


Total entries found: 1937
Scraping 1186 unique pages...


Scraping: 100%|██████████| 1186/1186 [03:47<00:00,  5.22it/s]


Done. Saved 1177 unique diseases to /content/drive/MyDrive/capstone/data/mayo_clinic_diseases_raw.csv
Columns: ['disease_name', 'symptoms', 'causes', 'risk_factors']
Failed pages: 0
  symptoms: 1168/1177 filled
  causes: 1164/1177 filled
  risk_factors: 1107/1177 filled
